# 🦟 Mosquito Breeding Ground Detection
## Comparing YOLOv8 · YOLOv5 · Detectron2 · DETR
**Paper:** *Mosquito Breeding Grounds Detection Using Deep Learning Techniques*  
**Authors:** Varalakshmi Perumal, R.Sasana, Rakshitha P, F.Joevita Faustina Doss  
**Dataset Classes:** `tires`, `water_tanks`, `bottles`, `buckets`, `pools`, `water_tubes`

## ⚙️ Step 0: Setup – Mount Drive & Clone Repo

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone project repository
import os
REPO_DIR = '/content/Mosquito-Breeding-Grounds-Detection'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Joevita20/Mosquito-Breeding-Grounds-Detection-.git {REPO_DIR}
%cd {REPO_DIR}

Mounted at /content/drive
Cloning into '/content/Mosquito-Breeding-Grounds-Detection'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 39 (delta 1), reused 38 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 5.15 MiB | 20.36 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Mosquito-Breeding-Grounds-Detection


## 📦 Step 1: Install Dependencies

In [3]:
!pip install -q ultralytics pypdf opencv-python-headless seaborn pycocotools timm scipy
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print('✅ All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 34.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 16.9 MB/s eta 0:00:00
✅ All dependencies installed


## 📥 Step 2: Download MBG Dataset
> Download from: https://www02.smt.ufrj.br/~tvdigital/database/mosquito/page_01.html
> Place videos in `/content/Mosquito-Breeding-Grounds-Detection/data/raw/videos/`

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VyiTP0iWE0793IW6xR5a")
project = rf.workspace("luis-augusto-silva-bq4bv").project("mosquito-suh0p")
version = project.version(4)
dataset = version.download("yolov8")

print(" Dataset downloaded!")
print("Dataset location:", dataset.location)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to mosquito-4 in yolov8:: 100%|██████████| 13520/13520 [00:01<00:00, 7217.44it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset downloaded!
Dataset location: /content/Mosquito-Breeding-Grounds-Detection/mosquito-4


In [5]:
import yaml

with open(f"{dataset.location}/data.yaml", "r") as f:
    data_info = yaml.safe_load(f)

print("Classes found:", data_info['names'])
print("Number of classes:", data_info['nc'])
print("Train path:", data_info.get('train'))
print("Val path:", data_info.get('val'))


Classes found: ['bucket', 'pool', 'tire', 'water tanks']
Number of classes: 4
Train path: ../train/images
Val path: ../valid/images


In [7]:
import os

data_dir = dataset.location
yaml_path = os.path.join(data_dir, "data.yaml")

yaml_content = f"""train: {data_dir}/train/images
val: {data_dir}/valid/images
test: {data_dir}/test/images

nc: 4
names:
  - bucket
  - pool
  - tire
  - water_tanks
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

# Verify all 4 classes are there
import yaml
with open(yaml_path) as f:
    check = yaml.safe_load(f)
print("Classes:", check['names'])
print("nc:", check['nc'])
print("Match:", len(check['names']) == check['nc'])


Classes: ['bucket', 'pool', 'tire', 'water_tanks']
nc: 4
Match: True


In [9]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    device=0,
    project='runs/yolov8',
    name='mbg_train',
    plots=True,
)

print(f"✅ Training done! Saved to: {results.save_dir}")


Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Mosquito-Breeding-Grounds-Detection/mosquito-4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mbg_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_

In [10]:
# Store YOLOv8 results for comparison later
v8_results = {
    'precision': 0.849,
    'recall': 0.701,
    'f1': round(2 * 0.849 * 0.701 / (0.849 + 0.701), 3),
    'map': 0.710,
}

# Per-class breakdown
v8_per_class = {
    'bucket':      {'mAP50': 0.298},
    'pool':        {'mAP50': 0.844},
    'tire':        {'mAP50': 0.703},
    'water_tanks': {'mAP50': 0.995},
}

print("✅ YOLOv8 Results stored!")
print(f"Overall mAP50: {v8_results['map']}")
print(f"F1-Score: {v8_results['f1']}")


✅ YOLOv8 Results stored!
Overall mAP50: 0.71
F1-Score: 0.768


In [8]:
# If using Roboflow export (YOLO format), paste your API snippet here:
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# project = rf.workspace().project('mbg-dataset')
# dataset = project.version(1).download('yolov8')

# OR: Extract frames from downloaded videos
!python src/preprocessing/extract_frames.py \
    --video_dir data/raw/videos \
    --output_dir data/frames \
    --fps 1


[INFO] Total frames extracted: 0


##  Step 3: Augment Dataset

In [ ]:
!python src/preprocessing/augment_data.py \
    --input_dir data/frames \
    --output_dir data/augmented \
    --n_augments 5
print(' Augmentation complete')

## Step 4: Train GAN for Minority Class (Pools)

In [ ]:
!python src/preprocessing/gan_gen.py \
    --real_dir data/augmented/pools \
    --output_dir data/gan_generated/pools \
    --num_images 200 \
    --epochs 200
print(' GAN generation complete')

##  Step 5: Convert to YOLO Format

In [ ]:
# If you have COCO-format annotations, convert them:
# !python src/preprocessing/convert_to_yolo_format.py \
#     --annotations data/raw/annotations.json \
#     --images_dir data/augmented \
#     --output_dir data/yolo_format

# Verify data.yaml
!cat data/data.yaml

## Step 6a: Train YOLOv8

In [ ]:
from ultralytics import YOLO

model_v8 = YOLO('yolov8n.pt')
results_v8 = model_v8.train(
    data='data/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    device=0,
    project='runs/yolov8',
    name='mbg_train',
    plots=True,
)
print(f'YOLOv8 training done. Saved to: {results_v8.save_dir}')

In [ ]:
# Validate YOLOv8
metrics_v8 = model_v8.val(data='data/data.yaml')
v8_results = {
    'precision': metrics_v8.box.mp,
    'recall': metrics_v8.box.mr,
    'f1': 2 * metrics_v8.box.mp * metrics_v8.box.mr / (metrics_v8.box.mp + metrics_v8.box.mr + 1e-8),
    'map': metrics_v8.box.map50,
}
print(f'YOLOv8 | Precision: {v8_results["precision"]:.3f} | Recall: {v8_results["recall"]:.3f} | mAP: {v8_results["map"]:.3f}')

## Step 6b: Train YOLOv5

In [ ]:
import subprocess, sys
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5
    !pip install -q -r yolov5/requirements.txt

!python yolov5/train.py \
    --data data/data.yaml \
    --weights yolov5s.pt \
    --epochs 100 \
    --batch-size 16 \
    --imgsz 640 \
    --optimizer SGD \
    --device 0 \
    --project runs/yolov5 \
    --name mbg_train

In [ ]:
# Validate YOLOv5
!python yolov5/val.py \
    --weights runs/yolov5/mbg_train/weights/best.pt \
    --data data/data.yaml \
    --device 0

##  Step 6c: Train Detectron2

In [ ]:
!python models/detectron2/train_detectron2.py \
    --data_dir data/detectron2_format \
    --optimizer adam \
    --epochs 60 \
    --device cuda

## Step 6d: Train DETR

In [ ]:
!python models/detr/train_detr.py \
    --data_dir data/detr_format \
    --epochs 140 \
    --device cuda

## Step 7: Compare All Models (Reproduce Table I from Paper)

In [ ]:
import sys
sys.path.insert(0, '.')
from src.evaluation.plot_results import plot_map_comparison, PAPER_RESULTS
from src.evaluation.metrics import print_metrics_table

# Fill these in with your actual validation results:
YOUR_RESULTS = {
    'YOLOv8':   v8_results,  # filled above
    # 'YOLOv5': {'precision': ..., 'recall': ..., 'f1': ..., 'map': ...},
    # 'Detectron2': {...},
    # 'DETR': {...},
}

print('\n📄 PAPER RESULTS (Table I):')
print_metrics_table(PAPER_RESULTS)

print('\n🧪 YOUR RESULTS:')
print_metrics_table(YOUR_RESULTS)

plot_map_comparison(YOUR_RESULTS, output_path='results/map_comparison.png')
from IPython.display import Image
Image('results/map_comparison.png')

##  Step 8: Run Inference on New Image

In [ ]:
from ultralytics import YOLO
from IPython.display import Image

model = YOLO('runs/yolov8/mbg_train/weights/best.pt')

# Change to any image path
TEST_IMAGE = 'data/yolo_format/test/images/your_image.jpg'
results = model.predict(TEST_IMAGE, conf=0.25, save=True, project='runs/inference')

# Show result
import glob
latest = sorted(glob.glob('runs/inference/**/*.jpg', recursive=True))[-1]
Image(latest)



```
# This is formatted as code
```

## Step 9: Save Results to GitHub

In [ ]:
!git add results/ runs/
!git commit -m 'Add model training results and comparisons'
!git push